In [1]:
from pathlib import Path
import logging
import datetime
import joblib
from itertools import product
import numpy as np
import numpy.typing as npt
from tqdm import tqdm
from src.module.utils import load_config, get_peak_trough, process_meanbp, load_npy
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar, minimize
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt

In [2]:
def set_vdb_environment(strategy, config, model_name) -> dict:
    data_path = Path(config["data_path"]).expanduser()
    src_path = data_path / "vitaldb" / "05.dataset_new"
    pos_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_pos_*.npy"))
    )
    neg_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_neg_*.npy"))
    )

    save_dir = Path(config["model_path"]).expanduser()
    save_dt = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
    save_path = save_dir / (save_dt + f"_{strategy}") / f"{model_name}.model.keras"
    env = {
        "src_path": src_path,
        "pos_path": pos_path,
        "neg_path": neg_path,
        "save_path": save_path,
    }
    return env


def set_mov_environment(strategy, config, model_name) -> dict:
    data_path = Path(config["data_path"]).expanduser()
    src_path = data_path / "mover" / "05.dataset_new"
    pos_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_pos_*.npy"))
    )
    neg_path = np.array(
        sorted(src_path.rglob(f"{strategy}_adj{config['adjacency']}_neg_*.npy"))
    )

    save_dir = Path(config["model_path"]).expanduser()
    save_dt = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
    save_path = save_dir / (save_dt + f"_{strategy}") / f"{model_name}.model.keras"
    env = {
        "src_path": src_path,
        "pos_path": pos_path,
        "neg_path": neg_path,
        "save_path": save_path,
    }
    return env


def vdb_load_dataset(env: dict) -> tuple[npt.NDArray, npt.NDArray]:
    pos_cid = load_npy(env["pos_path"][0])
    neg_cid = load_npy(env["neg_path"][0])
    pos_seg = load_npy(env["pos_path"][1])
    neg_seg = load_npy(env["neg_path"][1])
    pos_ts = load_npy(env["pos_path"][2])
    neg_ts = load_npy(env["neg_path"][2])
    pos_y, neg_y = np.ones(len(pos_seg)), np.zeros(len(neg_seg))
    X, y = np.vstack([pos_seg, neg_seg]), np.hstack([pos_y, neg_y])
    cid = np.hstack([pos_cid, neg_cid])
    ts = np.vstack([pos_ts, neg_ts])
    logging.info(f"load_dataset(pos/neg): {y.size}({pos_y.size}/{neg_y.size})")
    print(X.shape, y.shape, cid.shape, ts.shape)
    return cid, ts, X, y


def mov_load_dataset(env: dict) -> tuple[npt.NDArray, npt.NDArray]:
    pos_cid = load_npy(env["pos_path"][0])
    neg_cid = load_npy(env["neg_path"][0])
    pos_lt = load_npy(env["pos_path"][1])
    pos_seg = load_npy(env["pos_path"][2])
    neg_seg = load_npy(env["neg_path"][1])
    pos_ts = load_npy(env["pos_path"][3])
    neg_ts = load_npy(env["neg_path"][2])
    pos_y, neg_y = np.ones(len(pos_seg)), np.zeros(len(neg_seg))
    X, y = np.vstack([pos_seg, neg_seg]), np.hstack([pos_y, neg_y])
    cid = np.hstack([pos_cid, neg_cid])
    ts = np.vstack([pos_ts, neg_ts])
    logging.info(f"load_dataset(pos/neg): {y.size}({pos_y.size}/{neg_y.size})")
    print(X.shape, y.shape, cid.shape, ts.shape)
    return cid, ts, X, y


def split_array(arr, ratio, seed) -> tuple:
    train, test = train_test_split(arr, test_size=ratio, random_state=seed)
    train, val = train_test_split(train, test_size=ratio, random_state=seed)
    logging.info(
        f"total(train/val/test): {arr.size}({train.size}/{val.size}/{test.size}) cases"
    )
    return train, val, test


def extract_dataset(x, y, idx):
    extract_x = np.expand_dims(x[idx], 2)
    extract_y = y[idx]
    print(extract_y.shape)
    logging.info(
        f"X: {extract_x.shape} / y(pos/neg): {extract_y.size}({sum(extract_y==1)}/{sum(extract_y==0)})"
    )
    return extract_x, extract_y


def balance_dataset(X, y):
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) > len(neg_idx):
        pos_idx = np.random.choice(pos_idx, size=len(neg_idx), replace=False)
    else:
        neg_idx = np.random.choice(neg_idx, size=len(pos_idx), replace=False)

    pos_X, pos_y = X[pos_idx], y[pos_idx]
    neg_X, neg_y = X[neg_idx], y[neg_idx]
    res_X = np.vstack([pos_X, neg_X])
    res_y = np.hstack([pos_y, neg_y])
    print(res_y.shape)
    logging.info(
        f"X: {res_X.shape} / y(pos/neg): {res_y.size}({sum(res_y==1)}/{sum(res_y==0)})"
    )
    return res_X, res_y


def scale_input(input) -> npt.NDArray:
    return input.astype(np.float32) / 200

In [3]:
def get_min_size(x, p, t):
    if x[p].size < x[t].size:
        min_size = x[p].size
    else:
        min_size = x[t].size
    return min_size


def summary_stat(arr):
    x = arr.reshape(-1)
    x_mean, x_std, x_median, x_min, x_max = (
        np.mean(x),
        np.std(x),
        np.median(x),
        np.min(x),
        np.max(x),
    )
    return x_mean, x_std, x_median, x_min, x_max


def slope_intercept(t, y):
    t = np.asarray(t, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    tm, ym = t.mean(), y.mean()
    dt = t - tm
    denom = (dt * dt).sum()
    if denom == 0:
        return np.nan, np.nan
    slope = (dt * (y - ym)).sum() / denom
    return slope, ym - (slope * tm)


def extract_feature(arr, config):
    feature = []
    p, t = get_peak_trough(arr, config["detection"])
    min_size = get_min_size(arr, p, t)
    mbp = process_meanbp(arr.reshape(-1), config)
    sbp, dbp, pp = arr[p], arr[t], arr[p][:min_size] - arr[t][:min_size]
    mbp = np.unique(mbp[np.where(mbp != 0)])

    sbp_stat = summary_stat(sbp)
    dbp_stat = summary_stat(dbp)
    pp_stat = summary_stat(pp)
    mbp_stat = summary_stat(mbp)
    trend_per_beat, _ = slope_intercept(np.arange(mbp.size), mbp)
    trend_per_sec = trend_per_beat * (mbp.size / 30.0)
    half = mbp.size // 2
    slope_per_beat, _ = slope_intercept(np.arange(mbp.size)[half:], mbp[half:])
    slope_per_sec = slope_per_beat * (mbp.size / 30.0)
    last_mbp = mbp[-1]

    feature.extend(
        [
            *sbp_stat,
            *dbp_stat,
            *pp_stat,
            *mbp_stat,
            trend_per_sec,
            slope_per_sec,
            last_mbp,
        ]
    )
    return feature


def brier_skill_score(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    bs_model = brier_score_loss(y_true, y_prob)
    p = y_true.mean()  # 평가셋 자연 유병률
    bs_ref = p * (1.0 - p)  # 항상 p를 예측하는 무정보 모델
    return 1.0 - bs_model / bs_ref, bs_model, bs_ref

In [4]:
def calib_metrics(y, p):
    y = np.asarray(y, float)
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p))
    oe = y.sum() / p.sum()

    # slope: a + b*lp 동시 추정
    f = lambda t: -np.sum(y * (t[0] + t[1] * lp) - np.logaddexp(0, t[0] + t[1] * lp))
    slope = minimize(f, x0=[0.0, 1.0], method="L-BFGS-B").x[1]

    # intercept: slope를 1로 고정(offset)
    g = lambda a: -np.sum(y * (a + lp) - np.logaddexp(0, a + lp))
    icpt = minimize_scalar(g, bounds=(-10, 10), method="bounded").x

    return icpt, slope, oe


def patient_level_bootstrap(cid, seed, n, test_y, prob_y):
    # Patient-level bootstrap resampling
    pat_ids = np.unique(cid)
    idx_by_pat = {p: np.where(cid == p)[0] for p in pat_ids}
    np.random.seed(seed)  # 42

    auroc_list, auprc_list = [], []
    for i in tqdm(range(n), ncols=75):
        sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
        idx = np.concatenate([idx_by_pat[p] for p in sampled])
        sampled_test_y, sampled_prob_y = test_y[idx], prob_y[idx]

        auroc = roc_auc_score(sampled_test_y, sampled_prob_y)
        auprc = average_precision_score(sampled_test_y, sampled_prob_y)

        auroc_list.append(auroc)
        auprc_list.append(auprc)

    print("95% CI (without gray)")
    print(
        f"AUROC: ({np.percentile(auroc_list, 2.5):.3f}-{np.percentile(auroc_list, 97.5):.3f})"
    )
    print(
        f"AUPRC: ({np.percentile(auprc_list, 2.5):.3f}-{np.percentile(auprc_list, 97.5):.3f})"
    )

In [5]:
# VitalDB: 0 / MOVER: 1
config = load_config("../../config/conf.yaml")
vdb_path = Path(config["data_path"]).expanduser() / "vitaldb" / "06.train_val_test_new"
mov_path = Path(config["data_path"]).expanduser() / "mover" / "06.train_val_test_new"
model_path = Path(config["model_path"]).expanduser() / "revision_seed42"
strategy = "hypophetversion2"
vdb_env = set_vdb_environment(strategy, config, "none")
mov_env = set_mov_environment(strategy, config, "none")

In [6]:
cid, ts, X, y = vdb_load_dataset(vdb_env)
vdb_train_ucid, vdb_val_ucid, vdb_test_ucid = split_array(
    np.unique(cid), 0.2, config["seed"]
)
vdb_train_cid = cid[np.isin(cid, vdb_train_ucid)]
vdb_train_X, _ = extract_dataset(X, y, np.isin(cid, vdb_train_ucid))
vdb_train_y = np.full(len(vdb_train_X), 0)
vdb_val_cid = cid[np.isin(cid, vdb_val_ucid)]
vdb_val_X, _ = extract_dataset(X, y, np.isin(cid, vdb_val_ucid))
vdb_val_y = np.full(len(vdb_val_X), 0)
vdb_test_cid = cid[np.isin(cid, vdb_test_ucid)]
vdb_test_X, _ = extract_dataset(X, y, np.isin(cid, vdb_test_ucid))
vdb_test_y = np.full(len(vdb_test_X), 0)
print(vdb_train_cid.shape, vdb_train_X.shape, vdb_train_y.shape)
print(vdb_val_cid.shape, vdb_val_X.shape, vdb_val_y.shape)
print(vdb_test_cid.shape, vdb_test_X.shape, vdb_test_y.shape)

(226979, 3000) (226979,) (226979,) (226979, 3000)
(145764,)
(37159,)
(44056,)
(145764,) (145764, 3000, 1) (145764,)
(37159,) (37159, 3000, 1) (37159,)
(44056,) (44056, 3000, 1) (44056,)


In [7]:
cid, ts, X, y = mov_load_dataset(mov_env)
mov_train_ucid, mov_val_ucid, mov_test_ucid = split_array(
    np.unique(cid), 0.2, config["seed"]
)
mov_train_cid = cid[np.isin(cid, mov_train_ucid)]
mov_train_X, _ = extract_dataset(X, y, np.isin(cid, mov_train_ucid))
mov_train_y = np.full(len(mov_train_X), 1)
mov_val_cid = cid[np.isin(cid, mov_val_ucid)]
mov_val_X, _ = extract_dataset(X, y, np.isin(cid, mov_val_ucid))
mov_val_y = np.full(len(mov_val_X), 1)
mov_test_cid = cid[np.isin(cid, mov_test_ucid)]
mov_test_X, _ = extract_dataset(X, y, np.isin(cid, mov_test_ucid))
mov_test_y = np.full(len(mov_test_X), 1)
print(mov_train_cid.shape, mov_train_X.shape, mov_train_y.shape)
print(mov_val_cid.shape, mov_val_X.shape, mov_val_y.shape)
print(mov_test_cid.shape, mov_test_X.shape, mov_test_y.shape)

(25793, 3000) (25793,) (25793,) (25793, 3000)
(15828,)
(4853,)
(5112,)
(15828,) (15828, 3000, 1) (15828,)
(4853,) (4853, 3000, 1) (4853,)
(5112,) (5112, 3000, 1) (5112,)


In [17]:
train_cid = np.hstack([vdb_train_cid, mov_train_cid])
train_X = np.vstack([vdb_train_X, mov_train_X])
train_y = np.hstack([vdb_train_y, mov_train_y])
val_cid = np.hstack([vdb_val_cid, mov_val_cid])
val_X = np.vstack([vdb_val_X, mov_val_X])
val_y = np.hstack([vdb_val_y, mov_val_y])
test_cid = np.hstack([vdb_test_cid, mov_test_cid])
test_X = np.vstack([vdb_test_X, mov_test_X])
test_y = np.hstack([vdb_test_y, mov_test_y])
print(train_cid.shape, train_X.shape, train_y.shape)
print(val_cid.shape, val_X.shape, val_y.shape)
print(test_cid.shape, test_X.shape, test_y.shape)

(161592,) (161592, 3000, 1) (161592,)
(42012,) (42012, 3000, 1) (42012,)
(49168,) (49168, 3000, 1) (49168,)


In [18]:
trval_cid = np.hstack([train_cid, val_cid])
trval_X = np.vstack([train_X, val_X])
trval_y = np.hstack([train_y, val_y])
print(trval_cid.shape, trval_X.shape, trval_y.shape)

(203604,) (203604, 3000, 1) (203604,)


In [20]:
train_bal_X, train_bal_y = balance_dataset(train_X, train_y)
print(train_bal_X.shape, train_bal_y.shape)
trval_bal_X, trval_bal_y = balance_dataset(trval_X, trval_y)
print(trval_bal_X.shape, trval_bal_y.shape)

(31656,)
(31656, 3000, 1) (31656,)
(41362,)
(41362, 3000, 1) (41362,)


In [21]:
trval_bal_X_feat = np.array(
    [extract_feature(x, config) for x in tqdm(trval_bal_X, ncols=75)]
)
train_bal_X_feat = np.array(
    [extract_feature(x, config) for x in tqdm(train_bal_X, ncols=75)]
)
val_X_feat = np.array([extract_feature(x, config) for x in tqdm(val_X, ncols=75)])
test_X_feat = np.array([extract_feature(x, config) for x in tqdm(test_X, ncols=75)])
print(trval_bal_X_feat.shape, train_bal_X_feat.shape, val_X_feat.shape, test_X_feat.shape)

100%|██████████████████████████████| 49168/49168 [00:15<00:00, 3198.38it/s]

(41362, 23) (49168, 23)


# LR

In [ ]:
# best = (None, -np.inf)
# for c in [0.001, 0.01, 0.1, 1, 10, 100]:
#     pipe = Pipeline(
#         [
#             ("scaler", RobustScaler()),
#             (
#                 "clf",
#                 LogisticRegression(C=c, max_iter=1000, random_state=config["seed"]),
#             ),
#         ]
#     )
#     pipe.fit(train_bal_X_feat, train_bal_y)
#     prob_y = pipe.predict_proba(val_X_feat)[:, 1]
#     auroc = roc_auc_score(val_y, prob_y)
#     if auroc > best[1]:
#         best = (c, auroc)
# print(best)

(10, 0.6336595540195841)


In [ ]:
# pipe = Pipeline(
#     [
#         ("scaler", RobustScaler()),
#         (
#             "clf",
#             LogisticRegression(C=best[0], max_iter=1000, random_state=config["seed"]),
#         ),
#     ]
# )
# pipe.fit(train_bal_X_feat, train_bal_y)

Pipeline(steps=[('scaler', RobustScaler()),
                ('clf',
                 LogisticRegression(C=10, max_iter=1000, random_state=42))])

In [14]:
pipe = Pipeline(
    [
        ("scaler", RobustScaler()),
        (
            "clf",
            LogisticRegression(max_iter=1000, random_state=config["seed"]),
        ),
    ]
)
pipe.fit(trval_bal_X_feat, trval_bal_y)

Pipeline(steps=[('scaler', RobustScaler()),
                ('clf', LogisticRegression(max_iter=1000, random_state=42))])

In [15]:
prob_y = pipe.predict_proba(test_X_feat)[:, 1]
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.5, 1, 0)))

AUROC: 0.649
AUPRC: 0.186
Brier=0.237, Null Brier=0.093, BSS=-1.547
              precision    recall  f1-score   support

           0       0.93      0.60      0.73     44056
           1       0.15      0.60      0.24      5112

    accuracy                           0.60     49168
   macro avg       0.54      0.60      0.49     49168
weighted avg       0.85      0.60      0.68     49168



In [16]:
pat_ids = np.unique(test_cid)
idx_by_pat = {p: np.where(test_cid == p)[0] for p in pat_ids}
np.random.seed(config["seed"])

auroc_list, auprc_list = [], []
for i in tqdm(range(2000), ncols=75):
    vdb, mov = test_cid[test_y == 0], test_cid[test_y == 1]
    # stratified by dataset: vdb와 mov 환자가 균등하게 뽑히도록 하기 위함.
    sampled = np.hstack(
        [
            np.random.choice(np.unique(vdb), len(np.unique(vdb)), replace=True),
            np.random.choice(np.unique(mov), len(np.unique(mov)), replace=True),
        ]
    )
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_test_y, sampled_prob_y = test_y[idx], prob_y[idx]

    auroc = roc_auc_score(sampled_test_y, sampled_prob_y)
    auprc = average_precision_score(sampled_test_y, sampled_prob_y)

    auroc_list.append(auroc)
    auprc_list.append(auprc)

print("95% CI (without gray)")
print(
    f"AUROC: ({np.percentile(auroc_list, 2.5):.3f}-{np.percentile(auroc_list, 97.5):.3f})"
)
print(
    f"AUPRC: ({np.percentile(auprc_list, 2.5):.3f}-{np.percentile(auprc_list, 97.5):.3f})"
)

100%|██████████████████████████████████| 2000/2000 [00:28<00:00, 69.94it/s]

95% CI (without gray)
AUROC: (0.601-0.695)
AUPRC: (0.142-0.238)


# XGB

In [ ]:
# # tree-based ML은 스케일링이 필요없다.
# best = (None, -np.inf, None)
# grid = product([0.01, 0.05, 0.1], [3, 4, 6], [1, 5, 10])

# for lr, depth, mcw in grid:
#     clf = XGBClassifier(
#         n_estimators=1000,
#         learning_rate=lr,
#         max_depth=depth,
#         min_child_weight=mcw,
#         early_stopping_rounds=50,
#         random_state=config["seed"],
#     )
#     clf.fit(
#         train_bal_X_feat, train_bal_y, eval_set=[(val_X_feat, val_y)], verbose=False
#     )
#     auc = roc_auc_score(val_y, clf.predict_proba(val_X_feat)[:, 1])
#     if auc > best[1]:
#         best = ((lr, depth, mcw), auc, clf.best_iteration)

# print(best)

((0.1, 3, 10), 0.6575058599906051, 999)


In [ ]:
# lr, depth, mcw = best[0]
# clf = XGBClassifier(
#     n_estimators=1000,
#     learning_rate=lr,
#     max_depth=depth,
#     min_child_weight=mcw,
#     early_stopping_rounds=50,
#     random_state=config["seed"],
# )
# clf.fit(train_bal_X_feat, train_bal_y, eval_set=[(val_X_feat, val_y)], verbose=False)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=10, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

In [22]:
clf = XGBClassifier(random_state=config["seed"])
clf.fit(train_bal_X_feat, train_bal_y, eval_set=[(val_X_feat, val_y)], verbose=False)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [23]:
prob_y = clf.predict_proba(test_X_feat)[:, 1]
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.5, 1, 0)))

AUROC: 0.677
AUPRC: 0.200
Brier=0.228, Null Brier=0.093, BSS=-1.449
              precision    recall  f1-score   support

           0       0.93      0.63      0.75     44056
           1       0.16      0.62      0.26      5112

    accuracy                           0.63     49168
   macro avg       0.55      0.62      0.50     49168
weighted avg       0.85      0.63      0.70     49168



In [24]:
pat_ids = np.unique(test_cid)
idx_by_pat = {p: np.where(test_cid == p)[0] for p in pat_ids}
np.random.seed(config["seed"])

auroc_list, auprc_list = [], []
for i in tqdm(range(2000), ncols=75):
    vdb, mov = test_cid[test_y == 0], test_cid[test_y == 1]
    # stratified by dataset: vdb와 mov 환자가 균등하게 뽑히도록 하기 위함.
    sampled = np.hstack(
        [
            np.random.choice(np.unique(vdb), len(np.unique(vdb)), replace=True),
            np.random.choice(np.unique(mov), len(np.unique(mov)), replace=True),
        ]
    )
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_test_y, sampled_prob_y = test_y[idx], prob_y[idx]

    auroc = roc_auc_score(sampled_test_y, sampled_prob_y)
    auprc = average_precision_score(sampled_test_y, sampled_prob_y)

    auroc_list.append(auroc)
    auprc_list.append(auprc)

print("95% CI (without gray)")
print(
    f"AUROC: ({np.percentile(auroc_list, 2.5):.3f}-{np.percentile(auroc_list, 97.5):.3f})"
)
print(
    f"AUPRC: ({np.percentile(auprc_list, 2.5):.3f}-{np.percentile(auprc_list, 97.5):.3f})"
)

100%|██████████████████████████████████| 2000/2000 [00:27<00:00, 73.24it/s]

95% CI (without gray)
AUROC: (0.643-0.709)
AUPRC: (0.157-0.250)
